<a href="https://colab.research.google.com/github/divyachaudhary06/UCS420/blob/main/assignment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
roll_number = "1024160083"

# Last two digits
last_two_digits = [int(roll_number[-2]), int(roll_number[-1])]

categories = ["billing", "account", "general"]

# Fixed entries given in the assignment
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]


# Personalized entry for digit 8
digit = last_two_digits[0]
category = categories[digit % 3]

personalized_entry_1 = {
    "question": "how can i contact customer support",
    "answer": "You can contact customer support through the help desk.",
    "keywords": "support help contact assistance",
    "category": category
}


# Personalized entry for digit 3
digit = last_two_digits[1]
category = categories[digit % 3]

personalized_entry_2 = {
    "question": "how can i view my payment receipt",
    "answer": "You can view your payment receipt in the billing section.",
    "keywords": "receipt payment transaction billing",
    "category": category
}


# Combine all 6 entries
all_entries = fixed_entries + [
    personalized_entry_1,
    personalized_entry_2
]

df = pd.DataFrame(all_entries)

print("Q1: Final 6-row DataFrame")
print(df)

Q1: Final 6-row DataFrame
                             question  \
0              what is the annual fee   
1               how to reset password   
2         what are your working hours   
3               how can i pay the fee   
4  how can i contact customer support   
5   how can i view my payment receipt   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  You can contact customer support through the h...   
5  You can view your payment receipt in the billi...   

                              keywords category  
0                fee cost price charge  billing  
1                 password reset login  account  
2               hours timing open time  general  
3                  pay payment upi fee  billing  
4      support help contact assistance  gen

In [3]:
# Q2: Generate and Score a Hypothesis


def score_query(query, df):
    """
    Takes a query string and returns all matching
    entries ranked by confidence.
    """

    query_words = set(query.lower().split())

    results = []

    for index, row in df.iterrows():

        keyword_words = set(row["keywords"].lower().split())

        # Number of matching keywords
        matched_words = query_words.intersection(keyword_words)

        score = len(matched_words)

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    # Sort by highest score
    results.sort(key=lambda x: x["score"], reverse=True)

    return results


print("\nQ2: Scoring Example")

query = "how do i pay fee"

results = score_query(query, df)

for result in results:
    print(result)





Q2: Scoring Example
{'index': 3, 'question': 'how can i pay the fee', 'answer': 'You can pay via UPI, card, or net banking.', 'category': 'billing', 'score': 2}
{'index': 0, 'question': 'what is the annual fee', 'answer': 'The annual fee is Rs 500.', 'category': 'billing', 'score': 1}


In [5]:

# Q3: Find all questions belonging to a category


def same_category(category_name, df):
    """
    Return all questions belonging to a given category.
    """

    return df[df["category"] == category_name][
        ["question", "answer", "keywords", "category"]
    ]


print("\nQ3: FAQs in the personalized category 'general'")

result = same_category("general", df)

print(result)


Q3: FAQs in the personalized category 'general'
                             question  \
2         what are your working hours   
4  how can i contact customer support   

                                              answer  \
2                          We are open 9 AM to 5 PM.   
4  You can contact customer support through the h...   

                          keywords category  
2           hours timing open time  general  
4  support help contact assistance  general  


In [6]:

# Q4: Add a new keyword and save CSV


print("\nQ4: Add a new keyword")

# Pick one entry
entry_index = 0

print("Selected question:")
print(df.loc[entry_index, "question"])

new_keyword = input("Enter a new keyword: ")

# Add keyword to existing keywords
df.loc[entry_index, "keywords"] = (
    df.loc[entry_index, "keywords"] + " " + new_keyword
)

# Save updated DataFrame
filename = roll_number + "_faq_data.csv"

df.to_csv(filename, index=False)

print("\nUpdated DataFrame:")
print(df)

print("\nFile saved as:", filename)


Q4: Add a new keyword
Selected question:
what is the annual fee
Enter a new keyword: fine

Updated DataFrame:
                             question  \
0              what is the annual fee   
1               how to reset password   
2         what are your working hours   
3               how can i pay the fee   
4  how can i contact customer support   
5   how can i view my payment receipt   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  You can contact customer support through the h...   
5  You can view your payment receipt in the billi...   

                              keywords category  
0           fee cost price charge fine  billing  
1                 password reset login  account  
2               hours timing open time  general  
3       

In [7]:

# Q5: Count FAQ entries per category using groupby

print("\nQ5: Number of FAQ entries per category")

category_counts = df.groupby("category").size()

print(category_counts)


Q5: Number of FAQ entries per category
category
account    1
billing    3
general    2
dtype: int64


In [8]:
# ============================================================
# Q6: Modified scoring function with tie handling
# ============================================================

def score_query_with_ties(query, df):
    """
    Scores all matching entries.

    If multiple entries have the highest score,
    all tied entries are printed.
    """

    results = score_query(query, df)

    if len(results) == 0:
        print("No matching entries found.")
        return

    highest_score = results[0]["score"]

    top_matches = [
        result for result in results
        if result["score"] == highest_score
    ]

    print("\nQuery:", query)

    if len(top_matches) > 1:
        print("Tie detected! All highest-scoring matches:")

        for result in top_matches:
            print("\nQuestion:", result["question"])
            print("Answer:", result["answer"])
            print("Category:", result["category"])
            print("Score:", result["score"])

    else:
        print("Single best match:")

        result = top_matches[0]

        print("\nQuestion:", result["question"])
        print("Answer:", result["answer"])
        print("Category:", result["category"])
        print("Score:", result["score"])

In [9]:

# Q6 Demonstration 1: Query producing a tie


print("\nQ6: Tie demonstration")

score_query_with_ties("fee", df)


Q6: Tie demonstration

Query: fee
Tie detected! All highest-scoring matches:

Question: what is the annual fee
Answer: The annual fee is Rs 500.
Category: billing
Score: 1

Question: how can i pay the fee
Answer: You can pay via UPI, card, or net banking.
Category: billing
Score: 1


In [10]:

# Q6 Demonstration 2: Query without a tie


print("\nQ6: Non-tie demonstration")

score_query_with_ties("password reset", df)


Q6: Non-tie demonstration

Query: password reset
Single best match:

Question: how to reset password
Answer: Go to Settings > Reset Password.
Category: account
Score: 2
